In [365]:
import os
from pyoxigraph import Store, RdfFormat
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
from kneed import KneeLocator

from elasticsearch import Elasticsearch

from utils import ollama_request, EMBEDD_MODEL_1

import spacy
nlp = spacy.load("en_core_web_sm")

INDEX_NAME = os.getenv("INDEX_NAME")
ENT_INDEX_NAME = f"{INDEX_NAME}_entities_index"
FRAGMENT_INDEX_NAME = f"{INDEX_NAME}_fragments"
TRIPLETS_INDEX_NAME = f"{INDEX_NAME}_triplets_index"

EMBEDDING_MODEL = EMBEDD_MODEL_1

es_client = Elasticsearch('http://localhost:9200')

graph = Store()
graph.load(
    path=f"{INDEX_NAME}.ttl",
    format=RdfFormat.TURTLE
)

In [366]:
def extract_objects(text):
    doc = nlp(text)

    subjects = []
    target_deps = {
        "nsubj",
        "nsubjpass",
        "dobj",
        "pobj",
        "iobj",
    }
    for token in doc:
        if token.dep_ in target_deps:
            subject = " ".join(
                t.text for t in token.subtree
            )
            subjects.append(subject)

    return subjects

In [367]:
def execute_query(graph, start_date, end_date, filtering_criteria):
    query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ns1: <https://example.com/political-kg/property/>
PREFIX entity: <https://example.com/political-kg/entity/>
PREFIX predicate: <https://example.com/political-kg/predicate/>

SELECT ?s ?p ?o ?date ?start ?end ?speech_id ?triplet_id
WHERE {{
    ?stmt rdf:reifies <<( ?s ?p ?o )>> ;
          ns1:date ?date ;
          ns1:start ?start ;
          ns1:end ?end ;
          ns1:triplet_id ?triplet_id ;
          ns1:speech_id ?speech_id .

    FILTER (
        ?date > "{start_date}"^^xsd:date &&
        ?date < "{end_date}"^^xsd:date
        {filtering_criteria}
    )
}}
""".format(
        start_date=start_date,
        end_date=end_date,
        filtering_criteria=filtering_criteria
    )

    return graph.query(query)

In [368]:
def get_df_from_query_result(query_res):
    triplets_resp = {
        "subject": [],
        "predicate": [],
        "object": [],
        "date": [],
        "start": [],
        "end": [],
        "speech_id": [],
    }
    for row in query_res:
        fragment_start = row["start"].value
        fragment_end = row["end"].value
        speech_id = row["speech_id"].value

        triplets_resp["subject"].append(row["s"].value.split('/')[-1])
        triplets_resp["predicate"].append(row["p"].value.split('/')[-1])
        triplets_resp["object"].append(row["o"].value.split('/')[-1])
        triplets_resp["date"].append(str(row["date"].value))
        triplets_resp["start"].append(fragment_start)
        triplets_resp["end"].append(fragment_end)
        triplets_resp["speech_id"].append(speech_id)
    return pd.DataFrame(triplets_resp)

In [369]:
def get_triplets_by_id(
    graph,
    triplet_ids,
    start_date="1900-01-01",
    end_date="2100-01-01"
):
    triplet_ids_values = " ".join(str(x) for x in triplet_ids)

    query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ns1: <https://example.com/political-kg/property/>
PREFIX entity: <https://example.com/political-kg/entity/>
PREFIX predicate: <https://example.com/political-kg/predicate/>

SELECT ?s ?p ?o ?date ?start ?end ?speech_id
WHERE {{
    ?stmt rdf:reifies <<( ?s ?p ?o )>> ;
          ns1:date ?date ;
          ns1:start ?start ;
          ns1:end ?end ;
          ns1:speech_id ?speech_id ;
          ns1:triplet_id ?triplet_id .

    VALUES ?triplet_id {{ {triplet_ids_values} }}

    FILTER (
        ?date > "{start_date}"^^xsd:date &&
        ?date < "{end_date}"^^xsd:date
    )
}}
""".format(
        start_date=start_date,
        end_date=end_date,
        triplet_ids_values=triplet_ids_values,
    )

    return graph.query(query)

In [370]:
def knee_cutoff(scores):
    scores = np.sort(np.asarray(scores))[::-1]

    indices = np.arange(len(scores))

    knee = KneeLocator(
        indices,
        scores,
        curve="convex",
        direction="decreasing"
    )

    return int(knee.knee) - 1

In [371]:
def score_df_cutoff(df, top_k=10):
    df = df.iloc[:top_k, :].sort_values(by="score", ascending=False)

    sorted_scores = df["score"].values
    if len(sorted_scores) < 2:
        return df
    gaps = sorted_scores[:-1] - sorted_scores[1:]
    cutoff_1 = np.argmax(gaps) + 1
    cutoff_2 = knee_cutoff(sorted_scores)

    if cutoff_2 == 0:
        cutoff = cutoff_1
    else:
        cutoff = cutoff_2
    return df.iloc[:cutoff, :]

In [372]:
def find_similar_entities(entity, score_threshold=0.5, top_k=10):
    query_embedding = EMBEDDING_MODEL.encode(entity)
    final_res = {
        "score": [],
        "value_name": [],
    }
    response = es_client.search(
        index=ENT_INDEX_NAME,
        size=100,
        knn={
            "field": "value_embedding",
            "query_vector": query_embedding.tolist(),
            "k": 100,
            "num_candidates": 100,
            "filter": {
                "match_all": {}
            }
        },
        source=["value_name"]
    )

    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["value_name"].append(result["_source"]["value_name"])
    
    final_res = pd.DataFrame(final_res)
    return score_df_cutoff(final_res, top_k=top_k)


In [373]:
def find_question_related_triplets(graph, question, score_threshold=0.5, start_date="1900-01-01", end_date="2100-01-01", top_k=10):
    query_embedding = EMBEDDING_MODEL.encode(question)
    final_res = {
        "score": [],
        "triplet_id": [],
    }
    response = es_client.search(
        index=TRIPLETS_INDEX_NAME,
        size=1000,
        knn={
            "field": "embedding",
            "query_vector": query_embedding.tolist(),
            "k": 1000,
            "num_candidates": 1000,
            "filter": {
                "match_all": {}
            }
        },
        source=["triplet_id"]
    )

    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["triplet_id"].append(result["_source"]["triplet_id"])

    related_triplets = pd.DataFrame(final_res)
    related_triplets = score_df_cutoff(related_triplets, top_k=top_k)
    related_ids = list(related_triplets["triplet_id"])

    query_res = get_triplets_by_id(graph, related_ids, start_date=start_date, end_date=end_date)
    res_df = get_df_from_query_result(query_res)
    res_df["information_type"] = "main"

    return res_df


In [374]:
def find_additional_related_triplets(graph, question, score_threshold=0.5, top_entity_k=10, top_triplet_k=10, start_date="1900-01-01", end_date="2100-01-01"):
    question_objects = extract_objects(question)
    question_embedding = EMBEDDING_MODEL.encode(question)
    similar_entities = []
    for object in question_objects:
        similar_entities = similar_entities + list(find_similar_entities(object, score_threshold=score_threshold, top_k=top_entity_k)["value_name"].values)

    filtering_criteria = "&& (?s = entity:{val} || ?o = entity:{val})"
    triplets_resp = {
        "subject": [],
        "predicate": [],
        "object": [],
        "date": [],
        "start": [],
        "end": [],
        "speech_id": [],
        "triplet_id": [],
    } 
    for ent in similar_entities:
        try:
            query_res = list(
                execute_query(
                    graph=graph,
                    start_date=start_date,
                    end_date=end_date,
                    filtering_criteria=filtering_criteria.format(val=ent),
                )
            )
            # print(f"Found {len(query_res)} triplets for entity: {ent}")
        except Exception as e:
            continue

        for row in query_res:
            fragment_start = row["start"].value
            fragment_end = row["end"].value
            speech_id = row["speech_id"].value

            triplet_id = row["triplet_id"].value
            
            triplets_resp["subject"].append(row["s"].value.split('/')[-1])
            triplets_resp["predicate"].append(row["p"].value.split('/')[-1])
            triplets_resp["object"].append(row["o"].value.split('/')[-1])
            triplets_resp["date"].append(str(row["date"].value))
            triplets_resp["start"].append(fragment_start)
            triplets_resp["end"].append(fragment_end)
            triplets_resp["speech_id"].append(speech_id)
            triplets_resp["triplet_id"].append(int(triplet_id))

    triplets_df = pd.DataFrame(triplets_resp)
    final_res = {
        "score": [],
        "triplet_id": [],
    }

    response = es_client.search(
        index=TRIPLETS_INDEX_NAME,
        size=1000,
        knn={
            "field": "embedding",
            "query_vector": question_embedding.tolist(),
            "k": 1000,
            "num_candidates": 1000,
            "filter": {
                "terms": {
                    "triplet_id": triplets_df["triplet_id"].tolist()
                }
            }
        },
        source=["triplet_id"]
    )
    for result in response["hits"]["hits"]:
        score = result["_score"]
        if score >= score_threshold:
            final_res["score"].append(score)
            final_res["triplet_id"].append(result["_source"]["triplet_id"])

    final_scores = pd.DataFrame(final_res)
    final_scores["triplet_id"] = final_scores["triplet_id"].astype("int64")
    triplets_df["triplet_id"] = triplets_df["triplet_id"].astype("int64")

    triplets_df = triplets_df.merge(
        final_scores,
        on="triplet_id",
        how="inner"
    )
    triplets_df = score_df_cutoff(triplets_df, top_k=top_triplet_k)

    triplets_df["information_type"] = "additional"
    triplets_df = triplets_df.drop(columns=["score", "triplet_id"])
    return triplets_df

In [375]:
def get_prompt(graph, question, score_threshold=0.5, start_date="1900-01-01", end_date="2100-01-01"):

    main_triplets_related_to_question = find_question_related_triplets(
        graph=graph,
        question=question,
        score_threshold=score_threshold,
        start_date=start_date,
        end_date=end_date,
        top_k=1000
    )

    additional_triplets_related_to_question = find_additional_related_triplets(
        graph=graph,
        question=question,
        score_threshold=score_threshold,
        start_date=start_date,
        end_date=end_date,
        top_entity_k=10,
        top_triplet_k=1000
    )

    final_triplets = pd.concat([main_triplets_related_to_question, additional_triplets_related_to_question], ignore_index=True)

    final_triplets = final_triplets.drop_duplicates(subset=["subject", "predicate", "object", "date", "start", "end", "speech_id"])
    final_triplets = final_triplets.sort_values(by=["date", "start"], ascending=[True, True]).reset_index(drop=False)

    final_triplets["index"] = final_triplets.index + 1

    main_triplets = final_triplets[final_triplets["information_type"] == "main"]
    additional_triplets = final_triplets[final_triplets["information_type"] == "additional"]

    prompt = QUESTION_ANSWER_PROMPT.format(
        question=question,
        main_facts=main_triplets[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records"),
        additional_facts=additional_triplets[["index", "subject", "predicate", "object", "date"]].to_dict(orient="records")
    )

    return prompt, final_triplets

In [377]:
def get_fragment(triplets_resp, idx):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    start = int(row.start)
    end = int(row.end) + 1
    speech_id = row.speech_id
    return ".".join(es_client.search(index=INDEX_NAME, query={"match": {"_id": speech_id}}, size=100)["hits"]["hits"][0]["_source"]["text"].split(".")[start:end])

def get_speech(triplets_resp, idx):
    row = triplets_resp[triplets_resp["index"] == idx].iloc[0]
    speech_id = row.speech_id

    return es_client.search(index=INDEX_NAME, query={"match": {"_id": speech_id}}, size=100)["hits"]["hits"][0]["_source"]["text"]

In [378]:
QUESTION_ANSWER_PROMPT = """
You are a political scientist model. You are given a research question and a set of facts from a knowledge graph.
Each of facts is represented as a triplet in the form of (subject, predicate, object). The subject and object are entities, and the predicate describes the relationship between them.
Each of facts has a date associated with it, which indicates when the event described by the fact occurred. The date is represented in the format YYYY-MM-DD.
The triplets contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.
Each fact contains an index.
You can use the index to refer to specific facts when providing your answer by using syntax "some of your answer [index]".
Use just syntax with [index] to refer to the fact, do not use any additional words.

Facts are split into two categories: main and additional.
Main facts (most important) are directly related to the research question. 
Additional facts provide context or background information for entities present in the question.

Research question: {question}

Main Facts:
{main_facts}

Additional Facts:
{additional_facts}

Your task is to answer the research question based on the provided facts."
"""

In [379]:
QUESTION = """
What are Putin’s most common arguments for strengthening the army?
"""

START_DATE = "1999-01-01"
END_DATE = "2024-12-31"

PROMPT, final_triplets = get_prompt(
    graph=graph,
    question=QUESTION,
    score_threshold=0.5,
    start_date=START_DATE,
    end_date=END_DATE
)
print(f"Used main facts: {len(final_triplets[final_triplets['information_type'] == 'main'])}")
print(f"Used additional facts: {len(final_triplets[final_triplets['information_type'] == 'additional'])}")
PROMPT

Used main facts: 34
Used additional facts: 4


'\nYou are a political scientist model. You are given a research question and a set of facts from a knowledge graph.\nEach of facts is represented as a triplet in the form of (subject, predicate, object). The subject and object are entities, and the predicate describes the relationship between them.\nEach of facts has a date associated with it, which indicates when the event described by the fact occurred. The date is represented in the format YYYY-MM-DD.\nThe triplets contain information related to the research question, but they may not provide a complete answer. Your task is to analyze the facts and provide a concise answer.\nEach fact contains an index.\nYou can use the index to refer to specific facts when providing your answer by using syntax "some of your answer [index]".\nUse just syntax with [index] to refer to the fact, do not use any additional words.\n\nFacts are split into two categories: main and additional.\nMain facts (most important) are directly related to the researc

In [ ]:
res = ollama_request(
    prompt=PROMPT, is_stream=False
)
 
display(Markdown(res))

In [382]:
REFERENCE_NUM = 12

display(final_triplets[final_triplets["index"] == REFERENCE_NUM])#[["subject", "predicate", "object"]].reset_index(drop=True))

display(Markdown("### Reference Fragment"))
display(Markdown(get_fragment(final_triplets, REFERENCE_NUM)))

display(Markdown("### Reference Speech"))
display(Markdown(get_speech(final_triplets, REFERENCE_NUM)))


,index,subject,predicate,object,date,start,end,speech_id,information_type
11,12,vladimir_putin,introduced_military_reform,russian_armed_forces,2001-07-18,67,76,299,main


### Reference Fragment

 That is why we have developed a programme of modernisation of our Armed Forces, it is called the military reform, and it is to be implemented within 10 years. That programme devotes much attention to the development of the Navy.  I can say that I am not happy with the current state of the Navy. Nor am I happy about the number of combat assets in the Navy. I am absolutely sure that the Navy deserves more attention than it has enjoyed in recent years. It needs more attention compared with the other armed forces. But it does not mean that we should restore the amount of weaponry and level of weapons production that we had in the Soviet period. Not only are we unable to afford it, but it would be a waste of resources, effort and money. We must have an army that can absolutely ensure our defence capability, that is effective, lean, but not costly; we should minimise its cost, let us put it that way.  <…> The first part of your question was, who is Mr Putin? When was it, a year ago? Really, I would like to be spared the need to answer that question

### Reference Speech

Good afternoon, As you know, with the summer recess the active political season of the first half of the year has ended. However, many news agencies and individual journalists have recently applied for meetings and interviews – we have had more than a thousand requests. I have managed to meet with many of your colleagues and to talk with them. But of course I could not meet all these requests individually. So, my colleagues and I thought we would hold this kind of event to enable everyone to take part. And I would like to thank all those present for the interest you are showing in the political life of Russia and Russia in general. I will do my best to meet your curiosity and to answer all the questions you see fit to ask.  I think we might go straight into questions and answers not to waste time because we don’t have all that much time.  <…> You know, the main themes that will be discussed in Genoa are problems of a universal human character. In principle, the G8 leaders have agreed to focus on non-political issues – the fight against poverty and disease – which pose a particular threat to humankind. Then there are some other humanitarian, technological or man-made problems: alternative fuel, food safety, the human genome, and everything connected with the possible cloning of tissue or human beings and the moral aspects of that problem. As for political issues, and especially crisis issues, I think that these problems, including the Middle East, the conflicts in the Middle East and the problems in Iraq are sure to be discussed in bilateral contacts and perhaps in a free format on the fringes of the official meeting. You know our position on Iraq. I can repeat it. We know that the system of sanctions is counterproductive. The main thing today is to make sure whether Iraq is preparing to produce weapons of mass destruction, to be satisfied that this is not the case and, proceeding from that, to build our relations with Iraq. We must persuade the Iraqi leadership to allow observers to monitor the facilities that are of interest from the above-mentioned point of view and to move towards the lifting of sanctions. Endless extension of sanctions or their modification, in our opinion, will not achieve the goal we set ourselves. As regards the Middle East settlement and the conflict between Israel and Palestine, it is regrettable that the confrontation today has reached such a pitch. It is regrettable that the efforts exerted for many years by the former leadership of Israel and the Palestinian leadership have practically been annulled and the efforts of the international community have been annulled. I agree that today there is no more important task than to stop the violence on both sides. And it has to be done immediately. It is only on that basis that all the other issues can be tackled, and the problems of the region and between these two states are very serious. However, I think it would be very difficult to achieve unless the interests of all the countries in the region are taken into account. So, attention should be given to the legitimate demands and interests of, say, Syria; we should listen to what Egypt says and so on. Of course, we should move towards defusing tensions on the Israeli-Lebanese track.  We know the influence these countries have on various forces in the region. And I repeat, without taking into account the legitimate interests of these states we will not achieve a final solution of all these complex problems in the Middle East. Russia is ready to contribute to the settlement. And I would like to stress that we will try to make sure that none of the parties involved in the conflict suffers as a result of any agreements. That applies to Israel and to our Palestinian friends. Do you envy China or Russia? Let us take first things first. President Jiang Zemin and I have discussed the problem of possible withdrawal of the United States from the 1972 ABM Treaty, but it was not central to the visit. We discussed primarily our bilateral relations, the relations between the People’s Republic of China and the Russian Federation. As you know, we have signed a Treaty on Friendship and Cooperation. And it is indeed a very good basis for building inter-state relations.  The Russian Federation and China share a border that is thousands of kilometres long. We have still to regulate two sections of the border. I very much hope that during the year, after the consultations that we had with President Jiang Zemin in Beijing during my previous trip to China and more recently in Shanghai and now in Moscow, after our Foreign Ministries have stepped up work in this area, we will manage to come to terms on these two sections. That said, it is a complicated matter and I wouldn’t set any specific deadlines. We will continue negotiating until we reach a result that satisfies both sides.  As for the possible joint response of Russia and China to the US withdrawal from the 1972 AMB Treaty, every country decides for itself what to do, and how. Theoretically it is possible, but in practice Russia is not planning any joint actions in this sphere with other countries, including China. I must say that Russia today has sufficient strength and resources to react to any change in the sphere of international and strategic stability. We are ready for any turn of events.  As regards the territorial issues between Russia and Japan, I can say this. In my opinion, much has been done in this area in the last year and a half or two. Ultimately what matters most is how the people who consider that territory to be their homeland feel. That is, the people who consider it to be their homeland after the Second World War and those who considered it to be their homeland before the Second World War. We had to make steps that the leaders of our countries have spoken about for decades, but, on which, unfortunately, little has been done in practice.  The situation changed qualitatively in the last year and a half or two years. I would like to stress that. You know that Japanese citizens can freely enter that territory. Joint economic activities have begun there, and that creates a very good basis for a final solution of all the territorial issues. How it will be solved I cannot yet tell you. But I am sure that both Japan and the Russian Federation are very interested in solving it, and we will seek to resolve that problem.  You have spoken about better times. They were better times for some, but not for others. Everything is learnt by comparison. Yes, there was a period of time when we devoted a lot of time and invested almost all our resources in armaments, including the Navy. I am not going to pass judgment on whether it was good or bad. In any case, it is a fact that the arms race without regard for the real economic potential of the state was a major factor, though not the only factor, in undermining the economy and as a consequence, people’s trust in the government. We know this from the events in the early 1990s. I looked at it with dismay after I returned from abroad. I was aware of the emotional mood of the population, but I looked at all this with dismay. Of course, we cannot afford to allow a repetition of such events in our history. So, we should match our needs with our economic potential. That is why we have developed a programme of modernisation of our Armed Forces, it is called the military reform, and it is to be implemented within 10 years. That programme devotes much attention to the development of the Navy.  I can say that I am not happy with the current state of the Navy. Nor am I happy about the number of combat assets in the Navy. I am absolutely sure that the Navy deserves more attention than it has enjoyed in recent years. It needs more attention compared with the other armed forces. But it does not mean that we should restore the amount of weaponry and level of weapons production that we had in the Soviet period. Not only are we unable to afford it, but it would be a waste of resources, effort and money. We must have an army that can absolutely ensure our defence capability, that is effective, lean, but not costly; we should minimise its cost, let us put it that way.  <…> The first part of your question was, who is Mr Putin? When was it, a year ago? Really, I would like to be spared the need to answer that question. I wouldn’t like to characterise myself, let alone attach any labels, as you have asked me to. I think you and your colleagues can do a brilliant job of it without my help. Talented people, like all of you are, can do it very well. And in reality I think a person should not be judged by what he says about himself, but by what he does. Let us look and analyse what has happened in the political and economic spheres and in the development of the state.  Our state has the form of a federation. All the processes of stabilising the Russian state proceed under the country’s Constitution. And I would like to stress that it is happening within the framework of the existing Russian Constitution. If you look at the text of the Constitution, you will agree with me that it is one of the most democratic constitutions among civilised countries.  And we are developing our state on that basis without changing its main parameters. Just recently, yesterday in fact, I chaired the first meeting of the commission on the delimitation of powers between the federal centre, the regions and local government and I stressed once again that the commission should proceed strictly within the framework of the country’s Fundamental Law.  Secondly, it is clear that any young state, and because we have a new Constitution and a totally new system to that of the Soviet Union, modern Russia is a new state, in spite of its thousand-year-old history, any new institutions are in need of improvement. And of course, this is what we are doing.  I have already had occasion to speak about it. You can accept it or challenge it, but I will repeat my thesis again. I believe that during the 1990s the main provisions of the Constitution were flouted. And the gist of it was that a large number of regions hijacked some of the federal functions as a result of the passivity of the federal government. To restore justice in line with the country’s Fundamental Law, we have carried out some transformations: we changed the way the upper house of Parliament is formed, introduced federal districts and Presidential Envoys to the Federal Districts, whose main aim is to regain for the Federation the federal functions, which, I repeat, were to a large extent lost by the federal centre to the country’s regions.  Once that task had been fulfilled, we passed on to real modernisation of the economy and the political sphere. And there is no point in going into details, but I will recap on some of them.  In the political sphere, we have passed the law on parties. Again, one can argue whether it was necessary. I am convinced that it was. If civilised countries have de facto, and I stress, de facto, two, three or four functioning parties, why should Russia have 350 or 5,000? That is bacchanal and not democracy. The only result is that people are unable to determine their political sympathies. So, people choose not between ideologies, not between programmes, but between persons, between personalities. And it would have been that way in Russia if we hadn’t embarked on building a normal political base. That’s in the political sphere.  In the economic sphere, we are consistently seeking to curb bureaucracy in the economy. I must say that I had to, unfortunately – there is nothing wrong about it really – but still I had to call the Economy Minister in Parliament over some issues when he introduced documents, in particular, in the sphere of fighting bureaucracy in the economy, in the sphere of liberalisation of the economy and unreasonable state interference in the economy, as I have mentioned before.  Instead of 500-odd types of activities that require licensing we now have 102, if I am not mistaken. It was on that issue that I had to call the Economy Minister and back his efforts because, as you probably guess, a lot of bureaucrats of various levels, including in the Government, would like to continue issuing commands, permits, pieces of paper and so on.  At present, we have the most liberal tax policy anywhere. We have done a lot to combat bureaucracy and crime in the customs sphere. We fully meet our obligations to our foreign creditors and we are pursuing a balanced and peaceful foreign policy. We want to forge good-neighbourly relations with our neighbours and our main partners in Asia and in the West.  All these are part of Russian political reality of the past year and a half, and from this you can draw the conclusion as to who Mr Putin is. Finally, what would I like to see in three and a half years?  I would like what we have started and what we are aggressively promoting to come to fruition and bring real results that every ordinary citizen would feel in his pocket, so that Russian citizens feel safe, are more prosperous and happy and feel a growing sense of pride of their country.  I personally have never spoken of an adequate response to a possible NATO expansion. Why don’t you put this pointblank question to those who formulated the Russian position in that way? I have never formulated it that way. I can merely say that we do not see NATO as a hostile organisation and we do not regard its existence as a tragedy, although we don’t see why it should exist. It was born as an antipode to the Warsaw Pact, as an antipode to Soviet hegemony in Eastern Europe. Today there is no Warsaw Pact, no Soviet Union, but NATO exists and is growing successfully. And when we are told that it is a political organisation, that it is transforming itself from a military bloc into a political organisation, the question naturally suggests itself: why did it bomb Yugoslavia? Is that the job of a political organisation? Who did it? A military organisation did it and it does not make us feel happy. Another thesis. We constantly hear that everybody wants to bring down some kind of barriers and borders in Europe. We are all for it. But let us take a closer look at the implications. What does it mean to bring down borders and barriers? Let us think about it. And if it means pushing that barrier closer to the Russian borders, we are not very amused. Yes, those who are included in the common space will have no borders, but the borders are springing up in front of us. It results in different security levels on the continent and, in my opinion, it does not match the present-day realities and has not been prompted by any political or military exigencies.  Moreover, I can tell you with confidence that we will not achieve unity in Europe unless we create a common security and defence space. One can go about it in different ways. The simplest way is to disband NATO. But that is not on the agenda.  There is a second way. And by the way, I am not suggesting that we are in favour of that way, I am merely musing. The second possible variant is to admit Russia to NATO. That too creates a common defence and security space.  And the third variant is to create a new organisation that would perform these functions and in which Russia would be incorporated. That is a possible variant. And that is the task that was set before the OSCE. But today those who don’t seem to be too keen on seeing a common space and common security in Europe are giving a different tilt to the OSCE, directing it towards Central Asia and the North Caucasus or some other places, as long as it prevents it from building up the capacity and the potential for the sake of which it was created. But unless we do it some day, we will continue to have varying-level security in Europe and we will continue to mistrust each other. Having said that, I think it is clear to everyone that Russia is not threatening anyone and is not going to threaten anyone. Russia needs the rest of the civilised world and Europe, incidentally, just like Europe needs Russia. When we realise it and create corresponding structures, then the situation on the continent will change cardinally.  I think that the left in Russia have a future as a social-democratic idea. Moreover, Russia has a long tradition of the social-democratic movement, including the communist movement. The Communist Party of the Russian Federation is a legal party in the country. I have neither the moral, nor the legal right to declare from this podium that one of the political parties legally operating in the country has no future. It all depends on how the party’s leaders structure the activities of that organisation. I believe that if they embrace modern values, if they do not cling to what is no longer effective and what has discredited itself, the party certainly does have a future. If they fail to realise early on that times have changed, that the requirements to political organisations have changed, then they are in for some hard times.  As a first step I suggested at a meeting with the party’s leaders that historical justice be restored and the party revert to the initial name given it by its founder, Vladimir Lenin – the Russian Social-Democratic Labour Party. It would be a good sign and a move in the right direction. And regarding the burial of Lenin. I am against it. And I will explain why. The country has lived under the monopoly power of the CPSU for 70 years. It is the lifetime of a whole generation. Many people link their own lives with Lenin’s name. This is what Lenin’s burial would mean to them: to them it would mean that they had worshipped false values, that they had set themselves false tasks and that their lives had been wasted. We have many such people.  I think that the main achievement of the recent period is stabilisation and a measure of consensus in society. It is this consensus, and I would like to stress it, that helps us to promote the decisions that are vital for the modernisation of the political sphere and the economy. I cherish this and I will try not to do anything to upset the state of civil calm and consolidation in society. I think actions of this kind would produce an explosive situation which we have already experienced before, and it would impede real modernisation of Russia. But if we do it, if we change the economy and the political sphere, that would inevitably change the consciousness of the overwhelming majority of the people and then, proceeding from the new attitudes, I repeat, of the overwhelming majority of the population we will fulfil the will of the Russian people. And time will tell what the people’s will will be. <…> Do you feel the “sinister” impact of that document on you personally? Does it hinder you in any way? OK, the Information Security Doctrine has been adopted. Has it had a real impact on the activities of the media? I don’t think so. In general, every state keeps a close watch on this sphere. And this is particularly true of such countries as Russia because of the instability of its political system. In this country it is easy to sway public opinion through the media because people have few other benchmarks. Normal political parties are few and far between. Perhaps there are no large-scale national parties as yet apart from the Communist Party. There are some interest groups. And members of these groups fleet between parties, in a kind of Brownian movement.  But it is an extremely important sphere. I wouldn’t like any government actions to be prompted by the need to ensure security. In fact, one shouldn’t overuse the word. I don’t think the Information Security Concept is perfect. I am not going to criticise my colleagues, but some provisions and some language might have looked different.  As for information barriers separating us from our main partners, notably the Western countries, I simply do not see them, they do not exist. There are no barriers. As we see, leading news agencies, top journalists and your prominent colleagues from Russia and other countries are present here.  As a matter of fact, I would say that we are totally open, whereas we ourselves sometimes meet with obstacles. For instance, I am sure representatives of Radio Liberty are present here. Radio Liberty is operating here as a national radio station. But when our Ministry of the Press asked the US authorities to allow Radio Russia or Radio Mayak to operate on the same terms, we were turned down. So, if barriers are being set up, they are not being set up by us. That is one thing.  And secondly, I have some good news for you. I don’t know whether it is widely known, but VGTRK has become a co-founder of one of Europe’s largest information companies, Euronews. A group of our experts is going to Lyon. Beginning from September a Russian version of Euronews will be beamed live to Russia. Our audiences will get the full information provided by that news programme. And of course, all those who watch it in the world will watch news about Russia. I think it is a very good step forward in terms of integrating Russia in the European and world information space.  Can I take it as your official answer? Yes, your wish. We also have such a wish. Berezovsky who? (Animation in the audience, applause). You have kept referring to him as the former Secretary of the Security Council, former this or that. What is he now? A former deputy of the State Duma. But that too seems to have been forgotten. I have known Mr Berezovsky for a long time. He is an irrepressible and a very dynamic man, he is forever busy appointing someone or trying to topple someone. Let him work at it. In general, there is nothing wrong about it. You know the saying about fish: so that some fishes don’t go to sleep other fishes should badger them. In general, it is good if he finds things that we do wrong and makes it known to the public, we will only be too grateful to him for that because it would help us to adjust our own behaviour, and that is not bad. He is nobody’s fool, perhaps he will dig something up.  Now regarding Chechnya. That is a much more serious question and I will try to answer it seriously. I have spoken on this topic a great deal and I am sure you are familiar with what I said. Let me look at the matter from a slightly different angle.  The media people from Arab countries know that we witness a certain radicalisation of the Muslim world. And this worries many Muslim leaders and the leaders of former Soviet Union countries and some of our more distant colleagues and friends. One metastasis of that radicalisation has reached the North Caucasus.  You have mentioned Chechnya. And what was it that Chechnya didn’t like in 1999? It had been granted de facto independence. What made some armed people invade the territory of Dagestan and demand annexation from the Russian Federation of additional territories from the Caspian to the Black Sea and the formation of the United Islamic States? Was it prompted by the need to fight for the independence of Chechnya? We all understand that this is delirious rubbish, it has nothing in common with the interests of the Chechen people.  But once Russia realised that the territory could be used as a bridgehead to attack the Russian Federation and to destabilise it, Russia will know better than repeat the same mistake. Anyway, it will be an absolutely unpardonable mistake.  Without any doubt we must respect the opinion and the sentiments of the Chechen people. But we should not allow ourselves to be cheated as some radicals are trying to do. They are trying to supplant the interests of the Chechen people with their own fundamentalist aspirations and goals. And when we fight against them, they claim that we are fighting against Chechnya and its population, and some people take it up consciously or because they don’t understand the essence of the events. That is my position and I am not going to change it. Yes, I can. One tactic of the radical Islamists who are still trying to operate in Chechnya is to launch terrorist attacks on the federal forces, on the one hand, and, on the other hand, to provoke a retaliation and thus expose civilians so as to stir up the feelings of the civilians against the federal authorities. The so-called security raids you are referring to are aimed in fact at checking the identity of the people who are on the federal wanted list. I am not sure that the federal authorities have always been ready to resist the provocations of the militants.  I have said many times and I can repeat again: all violations of the law, all actions against civilians must be exposed and the culprits must be punished. As you know, until 1999 total chaos reigned in the Chechen Republic: people were executed in squares and people were beheaded. You ought to know it well. Thank God or thank Allah, we have put an end to it. At least you should thank us for it. That is one thing.  Secondly, the legal system there has been restored almost fully: there are functioning courts, the prosecutor’s office, notaries public and other services of the Justice Ministry have started working. Of course, after so many years of chaos and amid continuing terrorist attacks, it is practically impossible to restore order and create an effective system overnight. But we will persevere. We will bring to account both the militants who commit murder and our own citizens who do it.  You know that a year ago we were told: “You will never find a single Chechen in Chechnya who backs the efforts of the federal authorities.” And do you know that 40 heads of district administrations and imams and elders have been killed by militants? Why aren’t you asking me about it? Why aren’t you asking me how we are fighting these criminals?  If they are killed, it means that there are some Chechens who support us. Has it ever occurred to you to look at it from that angle? I suggest that you think about it. I repeat, unfortunately, now that a whole generation has grown up in a situation of total lawlessness, it is difficult to put things right overnight. But I repeat, we will act steadfastly, both by fighting those militants who break the law and those who yield to their provocations.  You know that many criminal cases have been opened, including against servicemen, and some of them get wide coverage in the media, and that is well known too. That must answer your question.  You know, in the States it is normal. The cliché jars a Russian ear a little bit. Whenever they speak about President Bush they refer to George Bush Jr. To me he is not junior. I was born in 1952 and he, I think, was born in 1946. To me he is above all a colleague who has worked in a region. I think I understand him very well in that sense. I myself worked for several years in a major region of the Russian Federation. I was Deputy Mayor of St Petersburg, and he was the head of one of the largest states in America. Maybe that was why it was easy for me to talk with him. Secondly, I must say that I do not share the opinion of those who think he is short of experience. During the year and a half as President of Russia I have tried to understand many things and I have tried to look deep into things. And I must say that in that sense he is quite a competent interlocutor with whom you can speak the same language and understand what we are talking about. And that also is a positive element because it is hard to talk to a person who has no grasp of certain things. I repeat, I had no problems there. He has done his homework.  And thirdly, on a purely human level – I think it is also an important thing, I don’t know if he will like it or not – but it seemed to me that he is a fairly warm person. He is pleasant to be with, I would say – and maybe I shouldn’t be saying it – but it seems to me that he is a little bit sentimental, but to me that is also a good sign. But at the same time he can put his foot down, especially on issues of international security. I have nothing more to add.  You know that we didn’t even try to reach agreement on some of the issues we discussed. We merely stated our positions. All that is a good basis on which to build personal relations and to try to find solutions to the tricky issues on which we have not yet come to terms.  <…> Indeed, a year has passed. It was a heavy moral blow for the Armed Forces, for the country as a whole, and for me too. My assessment of the events has not changed in any major way. I think that the rescue people did everything they could. If we now look at what was happening, stop-watch in hand, it can be said with absolute confidence that even if they had asked for foreign help in the very first second – although that was impossible because it was unclear what was happening, the military did not immediately understand what was happening, assistance wouldn’t have made it in time. Elementary timing of the events will demonstrate it.  Could the crew have been saved? Yes it could. But only if the people who had designed that type of submarine in the 1980s had foreseen such a development of the disaster and provided for corresponding rescue means. This was simply not built into the design. The standard rescue means were there and they were in normal condition and they were used. But, I repeat, if they had used all they had from the start and if they had asked for foreign help that very second, assistance would still have come too late. So, we should draw certain conclusions from it: technological and organisational. That goes without saying. But what I told you is a fact.  As far as my behaviour is concerned, in terms of PR, I should probably have gone back to Moscow. In essence there was no point in it because in Sochi and in Moscow I have the same amount of communication means at my disposal. I got the full information and I could act on the reports that I received.  So, in substance, it wouldn’t have made any difference. In terms of the coverage of events I might have demonstrated more zeal. But I repeat, unfortunately, it wouldn’t have made any substantive difference.  But today, first, I promised at a meeting in Vidyayevo that we will lift the submarine. I think we must do it and I’ll tell you why. One of the problems that this country has faced in recent years has been mistrust of the leadership. Trust can be restored only if we keep our promises: in economics, finances, in the political sphere – everywhere. You must be as good as your word. And that is important for the inner state of society.  Secondly, it must be done for environmental reasons. If it is possible to lift a reactor, it has to be done. Experts say it is possible. Repeated analyses of the situation, including the participation of foreign colleagues, show that it is feasible. That’s the second thing. And finally and thirdly, it is at least as important for the Navy and for the Armed Forces. We hope that after the lifting and during the examination of the submarine on the seabed additional data will be received to bring us closer to finding out the cause of the disaster, which, I repeat, is extremely important.  For all these reasons it has been decided to lift the submarine, and the operation has begun, as you know.  We have time for two or three more questions.  <…> I have spoken about it too. I think that personal relations between the leaders of such countries as the United States and Russia are very important if only because we have stockpiled the largest amounts of nuclear weapons, and in general it matters who you are dealing with. The feelings of that person are important. Even in a telephone conversation you feel the intonation. That already carries a certain message. That is why I welcome the fact that the President and I are developing normal personal relations.  As for our approaches to the problems of anti-missile defence, you know it well, and I will be repeating myself, but I would not like to do it. Needless to say the main criterion, and make no mistake about it, will be the interests of the national security of Russia. We will proceed from this above all. But of course on our part the proposals will be such that they will not upset international stability or undermine the system of international obligations in the sphere of strategic arms control that has taken shape over the past 20-odd years. We will move in that direction.  You know, when the decision was taken to create federal districts and appoint Presidential Envoys, fears were expressed that we would revert to over-centralisation. Of course, Russia has its own history of state building and its own history of administrative bodies, and unfortunately we do not have much experience of electing leaders. Perhaps at a certain stage we acted with undue haste, perhaps we shouldn’t have been in a hurry to introduce the election of the heads of regions. But since we have done it, I think it would be a big mistake to backtrack. First, people are used to deciding themselves who will be the number one on their territory.  Second. An elected executive, whether he wants it or not, bears a huge moral responsibility to his constituency, and that is a very important factor.  Finally and thirdly. The Russian Federation is a multi-national state. Russia has many ethnic entities: republics, areas, etc. We must be mindful of this. I am not sure that the population in some national republics would easily give up the right to elect their leader, and nothing could be worse than having some power bodies appointed and some elected. That is not a sound approach. We cannot ignore all these factors.  Moreover, in my opinion, considering the multi-national character of the Russian state, we must do everything we can so that the representative of every people, every nationality and every ethnic group, no matter how small, should feel comfortable. He should feel and realise that the place he lives in and Russia as a whole is his Motherland and that he won’t feel more comfortable anywhere else. In other words, there should be a set of issues on which decisions cannot be taken without the representatives of even the smallest people. That set of issues need not be very large or such as to impede the country’s development as a whole as a single state. But such issues must exist. I think that the right to elect their leader is one such issue. Finally, but equally importantly, we should not seek super-centralisation of the Soviet type. I think it is an ineffective system of government. And we must improve the system that we have put in place. I assume that much has yet to be done in order to delimit the responsibilities and powers and to back up these powers with financial resources. We will work in that direction.  Thank you for your attention. 